  <p align="center">
    <img
      src="../img/2025_01_02_NS_173_Snow_Survey.jpg"
      alt="CCSS snow survey at Phillips Station"
      width="1000"
      height="600"
    />
  </p>

  <p align="center">
    <span style="color: gray;"><em>
      Source: California Department of Water Resources. California Department of Water Resources staff members Manon von Kaenel (left), Jordan Thoennes, and Andy Reising conduct the first media snow survey of the 2025 season at Phillips Station in the Sierra Nevada, about 90 miles east of Sacramento off Highway 50 in El Dorado County. Photo taken January 2, 2025. Image obtained from
      <a href="https://pixel-ca-dwr.photoshelter.com/galleries/C0000lSOVLm_Sxso/G0000CdIWl5a2_xA/I0000OkEmK_kC73A/2025-01-02-NS-173-Snow-Survey-jpg">this link</a>.
    </em></span>
  </p>

# Retrieve California Cooperative Snow Surveys (CCSS) Data for a Watershed of Interest

  **Author:** Irene Garousi-Nejad ([igarousi@cuahsi.org](mailto:igarousi@cuahsi.org))
  **Last updated:** March 24, 2026

#### Introduction:  
This notebook demonstrates how to access the California Cooperative Snow Surveys (CCSS) data, which is program led by the California Department of Water Resources (DWR) to support water supply forecasting and flood management missions. 

### 1. Prepare the Python Environment

Ensure that the `cssi_evaluation` conda environment is selected as your Jupyter kernel. This environment should already be created if you followed the instructions under section "Creating your HydroLearnEnv Virtual Environment" in the `GettingStarted.md` file.

Import the libraries needed to run this notebook:

In [ ]:
import os
import time
import sys
from pathlib import Path

prefix = os.environ['CONDA_PREFIX']
os.environ['PROJ_LIB'] = os.path.join(prefix, 'share', 'proj')

# add the src directory to the path so we can import evaluation modules
#sys.path.append('../../src/')
repo_root = Path.cwd().resolve().parents[1]
sys.path.insert(0, str(repo_root / "src"))

import pyproj
import pandas as pd
import xarray as xr
import geopandas as gpd
from dask.distributed import Client

from cssi_evaluation.utils import plot_utils, dataPrep_utils
from cssi_evaluation.external_data_access import observation_utils

%load_ext autoreload
%autoreload 2

### 2. Set Inputs

In [ ]:
# repository path
repo_root = Path.cwd().resolve().parents[1]

# path to the model domain data
domain_data_path = f"{repo_root}/examples/nwm/domain_data/" 

# Path to the watershed shapefile
watershed_path = f"{domain_data_path}TolumneRiver_18040009.shp"

# Start and end times of a water year (note that this code currently works for one water year)
StartDate = '2018-10-01'
EndDate = '2020-09-30'

# Path to save results (obs and mod stands for observation and modeled, respectively)
OBS_OutputFolder = './cssi_outputs' 

### 3. Retrieve Observed Snow Data 

This code reads a GeoJSON file of all snow observation stations and filters them to include only stations in California with available CSV data. It also loads the watershed boundary shapefile (`TolumneRiver_18040009.shp`), converts it to the appropriate coordinate system, and merges all watershed polygons into a single MultiPolygon. Finally, it counts how many observation sites fall within this combined watershed boundary, providing the number of sites located inside the study area. 

**Heads up**: You might need to run the next cell twice. Sometimes, the spatial filtering does not return any results the first time due to how geometries are loaded or processed in the background.

In [ ]:
# Create geodataframe of all stations
all_stations_gdf = gpd.read_file('https://raw.githubusercontent.com/egagli/snotel_ccss_stations/main/all_stations.geojson').set_index('code')
all_stations_gdf = all_stations_gdf[all_stations_gdf['csvData']==True]
filtered_all_stations_gdf = all_stations_gdf[all_stations_gdf.state.str.contains('California')]  

# Read the watershed shapefile and standardize to WGS84
watershed_gdf = gpd.read_file(watershed_path).to_crs(epsg=4326)

# Combine all polygons into a single MultiPolygon
watershed_union = watershed_gdf.geometry.unary_union

# Use the polygon geometry to select snotel sites that are within the domain
gdf_in_bbox = filtered_all_stations_gdf[filtered_all_stations_gdf.geometry.within(watershed_union)]
gdf_in_bbox.reset_index(inplace=True)
print(f'There are {len(gdf_in_bbox)} sites from', ', '.join(set(gdf_in_bbox.network)), 'network(s) within the watershed.')


Plot these sites on a map. Then, hover over the pins to see the site names.

In [ ]:
gdf_in_bbox

In [ ]:
# Plot the sites within the watershed boundary using the plot_utils function
m = plot_utils.map_sites_within_watershed(gdf_in_bbox, watershed_gdf, zoom_start=9) 
m

Check the start and end date of available data for these sites.

In [ ]:
for i in gdf_in_bbox.index:
    print('Site ', gdf_in_bbox.iloc[i].code, ":", gdf_in_bbox.iloc[i].beginDate, "-", gdf_in_bbox.iloc[i].endDate)

The following uses the `observation_utils.py` script to download observed data for the sites within the domain. Since all the sites are from the (California Cooperative Snow Survey) CCSS network, we use the `getCCSSData` function from the module to get data. 

<div style="color:black; background-color:#f5f5f5; padding:10px; border-left: 5px solid #007acc;">
<h4>📖 Did you know?</h4>
<p>The California Cooperative Snow Surveys (CCSS) program, managed primarily by the California Department of Water Resources (DWR), monitors snowpack across key California watersheds to estimate annual water supply. Most CCSS sites are manual snow courses, where surveyors physically measure snow depth and snow water equivalent (SWE) several times a year, though some sites have been upgraded with automated sensors. In contrast, SNOTEL sites, managed by the USDA Natural Resources Conservation Service (NRCS), are fully automated and collect data continuously across the western United States, including California. While CCSS primarily supports water supply forecasting, SNOTEL supports both water supply forecasting and climate monitoring. </p>
</div>

In [ ]:
# Create a folder to save results
isExist = os.path.exists(OBS_OutputFolder)
if isExist == True:
    exit
else:
    os.mkdir(OBS_OutputFolder)

In [ ]:
# download CCSS data for each site within the watershed bounding box, save to /obs_outputs folder
for i in gdf_in_bbox.index:
    observation_utils.getCCSSData(gdf_in_bbox.name[i], gdf_in_bbox.code[i], 'Ca', StartDate, EndDate, OBS_OutputFolder)